# P5 (Disha): image–text checker on a Colab GPU

**First:** menu **Runtime → Change runtime type → T4 GPU → Save**.

Run the cells top to bottom (click a cell, press **Shift + Enter**). If Colab disconnects, run everything again from the top: finished pages are skipped.

## 1. Check the GPU
You should see a GPU name such as Tesla T4.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code
Downloads the repo and combines `main` with Aaron's renderer branch and your branch (once both are merged into `main`, this changes nothing).

In [ ]:
import os
REPO = "/content/LateComers-CSE576-NLP"
if not os.path.exists(REPO):
    !git clone -q https://github.com/aaryapendharkar/LateComers-CSE576-NLP.git {REPO}
%cd {REPO}
!git config user.email "p5@colab" && git config user.name "p5"
!git fetch -q origin
!git checkout -q -f -B p5-run origin/main
!git merge -q --no-edit origin/p4-renderer-v1 origin/p5-gap2-defense
!git log --oneline -3

## 3. Install packages (2–4 minutes)
Ignore red *dependency conflict* warnings unless the cell ends with an error.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y poppler-utils > /dev/null
!which pdftoppm
!pip install -q -r requirements-core.txt
!pip install -q -U transformers accelerate
import transformers; print("transformers", transformers.__version__)

## 4. Pick the model for this GPU
Writes `configs/local.yaml` (only on this machine). On a T4 it uses the smaller model from Aarya's config; on an A100 the main one. **Note which one ran**: it goes in the report's *Changes* section.

In [ ]:
import torch, yaml
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
big = mem_gb >= 30
local = {
    "pipelines": {"backend": "real"},
    "models": {
        "generator_fallback": None if big else "Qwen/Qwen3-VL-4B-Instruct",
        "device": "cuda",
        "dtype": "bfloat16" if torch.cuda.is_bf16_supported() else "float16",
    },
    "img_text_check": {"text_source": "pdfplumber", "max_new_tokens": 1024},
}
with open("configs/local.yaml", "w") as f:
    yaml.safe_dump(local, f)
print("model:", "Qwen/Qwen3.5-9B" if big else "Qwen/Qwen3-VL-4B-Instruct", f"({mem_gb:.0f} GB GPU)")

## 5. Build the pages (CPU, about 5 minutes)
Clean pages, Aaron's propagate pages, your conflict pages, then the text layer of every page.

In [ ]:
!python scripts/render_clean_pages.py --workers 2 2>&1 | tail -1
!python scripts/render_clean_pages.py --source chartqa --workers 2 2>&1 | tail -1
!python scripts/make_g2_propagation.py 2>&1 | grep -v "it/s"
!python scripts/make_g2_attacks.py 2>&1 | grep -v "it/s"
!python scripts/build_indexes.py --parse pdfplumber 2>&1 | tail -1

## 6. Dry run with a perfect reader (seconds, no model)
Checks the text-layer side. Expected: conflict pages ~100% detected, propagate ~0%, clean pages ~0–2% flagged.

In [ ]:
!python scripts/eval_img_text_check.py --oracle --overwrite 2>&1 | grep -v 'it/s'

## 7. Model test on 5 pages (first run downloads the model, 5–10 minutes)
Then shows what the model read next to what is really on the page. **If the reads look wrong, paste this output to Claude.**

In [ ]:
!python scripts/eval_img_text_check.py --n-clean 50 --limit 5 --read-accuracy 2>&1 | grep -v 'it/s'

In [ ]:
import json
for r in map(json.loads, open("results/gap2/checker_reads.jsonl")):
    print(r["page_id"], "| read exactly:", r["exact"], "| misreads:", r["misreads"][:3])

## 8. Full run (about 1 hour on a T4)
50 clean table pages, 50 clean chart pages and every Gap 2 page. Resumable: if Colab disconnects, run cells 2–5 again, then this cell.

In [ ]:
!python scripts/eval_img_text_check.py --n-clean 50 --read-accuracy 2>&1 | grep -v 'it/s'

In [ ]:
print(open('results/gap2/checker_table.md').read()); print(open('results/gap2/checker_reads.md').read())

## 9. Save to Google Drive
Colab deletes everything when it disconnects. This copies the results and the saved model reads to **My Drive / cse576_p5**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/cse576_p5
!cp -r results/gap2 /content/drive/MyDrive/cse576_p5/
!cp -r data/parsed/img_tables /content/drive/MyDrive/cse576_p5/
!ls /content/drive/MyDrive/cse576_p5/gap2

## 10. Optional: put the saved reads back after a disconnect
Run this after cells 2–5 on a new machine, before cell 8, so pages already read are not read again.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p data/parsed results
!cp -r /content/drive/MyDrive/cse576_p5/img_tables data/parsed/ 2>/dev/null; cp -r /content/drive/MyDrive/cse576_p5/gap2 results/ 2>/dev/null; echo restored